# QUENCH — simulations and figures

**Uptake, Engagement, Normalization, Coverage, Handover** — the effector half of BOREAS.
This notebook reproduces every simulation, table and figure in the QUENCH model document
from the equations as printed.

Run the cells in order. **Runtime → Run all** works; the whole notebook takes about
3–5 minutes, almost all of it in the global sensitivity analysis at the end.

### What is in here

| Section | Produces |
|---|---|
| 1 · Setup | plotting style, imports |
| 2 · Parameters | FLARE upstream constants + the §3 QUENCH table |
| 3 · Equations | Eqs 1–8 as one coupled ODE system, integrated with LSODA |
| 4 · Verification | every published number, checked against the document |
| 5 · Delivery chain | §2.2 — the degeneracy proof, the escape table, the 17% requirement |
| 6 · Figure 1 | compartments and RISC loading |
| 7 · Figure 2 | transcript and protein trajectories for both targets |
| 8 · Figure 3 | secretion sweep |
| 9 · Severity | Eq 10, the E_max ceiling and the coverage table |
| 10 · Figure 4 | Scenario-A open-loop handover |
| 11 · Figure 5 | conditional closed loop |
| 12 · Figure 6 | local sensitivity |
| 13 · Figure 7 | global sensitivity (Latin hypercube, seed 20260910) |

Every figure is also written to a `.png` next to the notebook, so you can download them
from the Colab file browser on the left.

> **One note on what this notebook is.** It reproduces the model exactly as documented.
> It does not re-derive the assumptions underneath it — the escape fraction of 0.75,
> the calibrated k₇, and the assumed 250 µM FLARE input all enter here as given, and
> the document is explicit that they are unmeasured. The numbers below are conditional
> on them.

## 1 · Setup

In [ ]:
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from scipy.stats import qmc, spearmanr

# BOREAS house palette (sampled from the published figures)
BLUE, DARK, RED, TAN = '#005389', '#1a2733', '#9c3b28', '#d39b5e'
MID, FILL, GREY      = '#5f9fc4', '#cceaf5', '#9aa7b1'

plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 11, 'figure.dpi': 110,
    'text.color': DARK, 'axes.labelcolor': DARK, 'xtick.color': DARK,
    'ytick.color': DARK, 'axes.edgecolor': '#b9c6d0', 'axes.linewidth': 0.8,
    'grid.color': '#dee8ef', 'grid.linewidth': 0.9, 'axes.grid': True,
    'axes.axisbelow': True, 'legend.frameon': False, 'figure.facecolor': 'white',
})

SAVE_PNG = True          # also write each figure to disk

def show(fig, name):
    if SAVE_PNG:
        fig.savefig(f'{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

print('ready')

## 2 · Parameters

Two sets. `P1` is the upstream FLARE circuit, integrated alongside QUENCH so that
the promoter trajectory **P_trxB(t)** is produced rather than assumed — this is the
interface the document describes as "the full trajectory, not a plateau value".

`Q` is the QUENCH parameter table from §3.

**On α_sec.** The document is careful here and so is this code: α_sec = 6.14 nM h⁻¹ is
held **fixed** in every forward run. The familiar 4.83 nM h⁻¹ is the *product*
α_sec × P_trxB at FLARE's assumed 250 µM condition, not a separate constant — it moves
whenever the upstream trajectory moves.

**On R_total.** The §3 table prints 16.6 nM; the published figures were computed at
16.61. The difference is 0.01 percentage points on the plateau, so either reproduces
the document. 16.61 is used here to match the figures exactly.

In [ ]:
# ---- FLARE upstream circuit -------------------------------------------------
P1 = dict(f_in=0.107, chi_OX=0.40, PerR_tot=6.0, k_Fe=12.5, phi_Fe=0.05,
          delta_Per=0.462, K_perR=1.159, n_perR=1.212, k_clp=6.0, min_clp=0.097,
          K_clp=1.5, n_clp=2.0, k_syn_spx=3.0, delta_spx=0.35, k_ox=0.060,
          k_trx=1.20, Trx_bas=0.50, Trx_max=0.90, k_trx_rel=2.0,
          P_basal=0.010, K_out=0.60, n_out=1.0)
KM_NPX, VMAX_FLARE = 28.0, 60.3      # FLARE clearance constants (Eqs 1-2 upstream)
H_SS = 250.0                         # assumed lesional H2O2 -- see the document's caveat

# ---- QUENCH parameter table (section 3) -------------------------------------
Q = dict(
    k_eff = 0.0263,   # extracellular disappearance          h^-1   EST
    k1    = 0.15,     # uptake into keratinocytes            h^-1   EST
    k2    = 0.30,     # GALA-enhanced endosomal escape       h^-1   EST
    k3    = 0.10,     # endosomal degradation                h^-1   EST (no source)
    a     = 1.0,      # processing to mature siRNA           h^-1   EST (no source)
    k4    = 0.029,    # cytoplasmic loss before processing   h^-1   EST
    k5    = 0.001,    # RISC loading                      nM^-1h^-1 EST
    k6    = 0.029,    # free siRNA degradation               h^-1   EST
    Rtot  = 16.61,    # available Argonaute capacity          nM    EST
    kA    = 0.077,    # active RISC turnover                 h^-1   EST
    k7    = 0.010,    # RISC-transcript association       nM^-1h^-1 CAL (no source)
    k8    = 7.2,      # cleavage and release                 h^-1   EST
    k10   = 0.10,     # transcript turnover                  h^-1   CAL
    m0    = 1.0,      # normalised baseline transcript      nM-eq   EST
)
ALPHA = 6.14                                   # nM h^-1, held fixed  (CAL)

STAT6_AD, RelA_AD = 169.33, 48.18              # CPM, lesional   (LIT, GSE121212)
S6_H, RA_H        = 157.893294, 38.493335      # CPM, healthy    (LIT, GSE121212)

E_S6, E_RA = STAT6_AD / S6_H, RelA_AD / RA_H   # elevation ratios
REQ_S6 = 100 * (1 - 1 / E_S6)                  # knockdown needed to reach healthy
REQ_RA = 100 * (1 - 1 / E_RA)

print(f'STAT6 elevation {E_S6:.4f}x  -> requires {REQ_S6:.2f}% knockdown')
print(f'RelA  elevation {E_RA:.4f}x  -> requires {REQ_RA:.2f}% knockdown  <- binding constraint')

## 3 · The equations

Eqs 1–8 as one coupled system. The state vector is 15 long: four FLARE variables
(PerR, Spx_red, Spx_ox, Trx) followed by eleven QUENCH variables (E, N, B, then the
two siRNA branches, two active-RISC branches, two complexes and two transcripts).

The two branches are written out separately rather than vectorised, so each line
maps to a term you can find in the document.

`t_off` forces the promoter to zero at a given time — that is the Scenario-A
handover used in Figure 4, and it is a *construct*, not a property of the equations.

In [ ]:
def hill(x, K, n):
    x = np.maximum(x, 0.0)
    return x**n / (K**n + x**n)

def inputs(H, p=P1):
    """Applied H2O2 -> intracellular peroxide and lumped oxidant field (FLARE Eq 3)."""
    return p['f_in'] * H, p['chi_OX'] * H

def rhs(t, y, H, Pt_gate, q, alpha, p=P1):
    PerR, Sr, So, Trx, E, N, B, S1, S2, A1, A2, C1, C2, m1, m2 = y
    I_int, OX = inputs(H, p)

    # ---- FLARE: the promoter that drives secretion ---------------------------
    P_spx = 1.0 / (1.0 + (max(PerR, 0) / p['K_perR'])**p['n_perR'])
    g     = 1.0 - (1.0 - p['min_clp']) * hill(I_int, p['K_clp'], p['n_clp'])
    deg   = p['k_clp'] * g + p['delta_spx']
    v_ox  = p['k_ox']  * OX  * Sr
    v_re  = p['k_trx'] * Trx * So
    Pt    = (p['P_basal'] + (1 - p['P_basal'])
             * hill(So, p['K_out'], p['n_out'])) * Pt_gate

    # ---- QUENCH ---------------------------------------------------------------
    Rf = max(0.0, q['Rtot'] - A1 - A2 - C1 - C2)     # Eq 4, shared Argonaute pool
    k9 = q['k10'] * q['m0']                          # Eq 7, synthesis at baseline

    return [
        # FLARE
        p['delta_Per']*(p['PerR_tot'] - PerR) - p['phi_Fe']*p['k_Fe']*I_int*PerR,
        p['k_syn_spx']*P_spx - deg*Sr - v_ox + v_re,
        v_ox - v_re - deg*So,
        p['k_trx_rel']*(p['Trx_bas'] + (p['Trx_max'] - p['Trx_bas'])*Pt - Trx),
        # Eq 1  extracellular pool
        alpha*Pt - (q['k1'] + q['k_eff'])*E,
        # Eq 2  endosomal, cytoplasmic, and the two mature siRNA branches
        q['k1']*E - (q['k2'] + q['k3'])*N,
        q['k2']*N - (q['a']  + q['k4'])*B,
        q['a']*B - q['k5']*S1*Rf - q['k6']*S1,
        q['a']*B - q['k5']*S2*Rf - q['k6']*S2,
        # Eq 5  active RISC, per branch
        q['k5']*S1*Rf - q['k7']*A1*m1 + q['k8']*C1 - q['kA']*A1,
        q['k5']*S2*Rf - q['k7']*A2*m2 + q['k8']*C2 - q['kA']*A2,
        # Eq 6  RISC-transcript complex
        q['k7']*A1*m1 - q['k8']*C1,
        q['k7']*A2*m2 - q['k8']*C2,
        # Eq 7  target transcripts
        k9 - q['k10']*m1 - q['k8']*C1,
        k9 - q['k10']*m2 - q['k8']*C2,
    ]

def run(alpha=ALPHA, t_end=336.0, n=12000, q=None, t_off=None, H=H_SS):
    """Integrate FLARE+QUENCH. t_off forces the promoter to zero (Scenario-A handover)."""
    qq = {**Q, **(q or {})}
    def f(t, y):
        gate = 1.0 if (t_off is None or t <= t_off) else 0.0
        return rhs(t, y, H, gate, qq, alpha)
    y0 = [P1['PerR_tot'], 0, 0, P1['Trx_bas'], 0, 0, 0, 0, 0, 0, 0, 0, 0,
          qq['m0'], qq['m0']]
    s = solve_ivp(f, (0, t_end), y0, t_eval=np.linspace(0, t_end, n),
                  method='LSODA', rtol=1e-9, atol=1e-12)
    t, So = s.t, s.y[2]
    Pt = P1['P_basal'] + (1 - P1['P_basal'])*hill(So, P1['K_out'], P1['n_out'])
    if t_off is not None:
        Pt = np.where(t <= t_off, Pt, 0.0)
    return dict(t=t, P_trxB=Pt, E=s.y[4], N=s.y[5], B=s.y[6],
                S=s.y[7], A=s.y[9], C=s.y[11],
                Rfree=qq['Rtot'] - s.y[9] - s.y[10] - s.y[11] - s.y[12],
                m1=s.y[13], m2=s.y[14],
                STAT6=s.y[13]*STAT6_AD, RelA=s.y[14]*RelA_AD,
                cum=np.concatenate([[0], np.cumsum(np.diff(t)*alpha*Pt[:-1])]))

def at(r, key, t):
    return float(np.interp(t, r['t'], r[key]))

def cross(r, key, level):
    """First time a trajectory crosses a level (linear interpolation)."""
    y = r[key]
    for i in range(1, len(y)):
        if (y[i-1] - level) * (y[i] - level) <= 0:
            return float(np.interp(level, [y[i], y[i-1]], [r['t'][i], r['t'][i-1]]))
    return None

def plateau(q=None, alpha=ALPHA, T=600.0):
    """Fractional knockdown at steady state, in percent."""
    return 100 * (1 - at(run(alpha=alpha, q=q, t_end=T, n=8000), 'm1', T))

def with_escape(e):
    """Return a parameter override that sets the escape fraction k2/(k2+k3) to e."""
    return {'k3': Q['k2'] * (1 - e) / e}

print('equations defined')

## 4 · Verification

Thirteen published quantities, each recomputed from the equations above. This is the
cell to run first if you change anything — if a row turns to `XX`, the change broke
something.

In [ ]:
r = run()

checks = [
    ('Free siRNA per branch (Fig 1)',    at(r, 'S', 336),                  88,    'nM', 1.5),
    ('Active RISC per branch (Fig 1)',   at(r, 'A', 336),                  5.77,  'nM', 0.05),
    ('Free Argonaute remaining',         at(r, 'Rfree', 336),              5.1,   'nM', 0.1),
    ('Plateau knockdown',                100*(1 - at(r, 'm1', 336)),       36.6,  '%',  0.2),
    ('STAT6 crosses healthy',            cross(r, 'STAT6', S6_H),          20.3,  'h',  0.3),
    ('RelA crosses healthy',             cross(r, 'RelA',  RA_H),          32.7,  'h',  0.3),
    ('Active RISC at RelA endpoint',     at(r,'A',cross(r,'RelA',RA_H)),   3.73,  'nM', 0.05),
    ('CO-BERA released at endpoint',     at(r,'cum',cross(r,'RelA',RA_H)), 155,   'nM', 3),
    ('STAT6 settles at',                 at(r,'m1',336)*E_S6,              0.68,  'x',  0.01),
    ('RelA settles at',                  at(r,'m2',336)*E_RA,              0.79,  'x',  0.01),
    ('Plateau at 8x secretion',          plateau(alpha=ALPHA*8),           44.2,  '%',  0.3),
    ('Plateau at 1/8x secretion',        plateau(alpha=ALPHA/8),           13.4,  '%',  0.3),
    ('72 h local-sensitivity baseline',
        100*(1 - at(run(t_end=120, n=4000), 'm1', 72.0)),                  34.6,  '%',  0.2),
]

print(f'{"":4}{"Quantity":<34}{"model":>9}{"document":>12}')
print('  ' + '-'*57)
for name, got, want, unit, tol in checks:
    flag = 'ok ' if abs(got - want) <= tol else 'XX '
    print(f'  {flag}{name:<34}{got:>8.2f} {unit:<3}{want:>8} {unit}')

## 5 · The delivery chain — §2.2

The central structural result. Each forward step competes with a loss from the same
pool, so at steady state only the **ratio** reaches the outcome. Six constants collapse
to three efficiencies, and their product is the fraction of secreted CO-BERA that
becomes usable siRNA.

The degeneracy is not approximate. Scaling a numerator and its denominator together
by any factor leaves the plateau unchanged to four decimal places.

The escape fraction is the one that matters, and it is the one with no matched
measurement. The cell below inverts the model to state the **requirement** instead:
the escape fraction each target needs in order to be brought back to healthy.

In [ ]:
up  = Q['k1'] / (Q['k1'] + Q['k_eff'])
esc = Q['k2'] / (Q['k2'] + Q['k3'])
pro = Q['a']  / (Q['a']  + Q['k4'])

print('Delivery chain')
print(f"  uptake      k1/(k1+k_eff) = {up:.3f}")
print(f"  escape      k2/(k2+k3)    = {esc:.3f}   <- unmeasured for L. plantarum EVs")
print(f"  processing  a/(a+k4)      = {pro:.3f}")
print(f"  product                   = {up*esc*pro:.3f}")

print('\nDegeneracy proof - scale each pair together, plateau must not move')
for x, y in [('k1', 'k_eff'), ('k2', 'k3'), ('a', 'k4')]:
    vals = [plateau({x: Q[x]*L, y: Q[y]*L}) for L in (0.25, 1.0, 4.0)]
    print(f"  {x:>2} x {y:<6} " + '  '.join(f'{v:8.4f}' for v in vals)
          + f"   spread {max(vals)-min(vals):.4f} pp")

print('\nEscape fraction -> outcome')
rows = []
for e, label in [(0.75, 'nominal assumption'),
                 (0.15, 'illustrative stress-test'),
                 (0.02, 'LNP comparator [18]')]:
    p = plateau(with_escape(e))
    rows.append(dict(escape=f'{e:.2f}', label=label, chain=round(up*e*pro, 4),
                     plateau=f'{p:.1f}%',
                     meets_RelA='yes' if p > REQ_RA else 'no'))
print(pd.DataFrame(rows).to_string(index=False))

req = lambda tgt: brentq(lambda e: plateau(with_escape(e)) - tgt, 0.005, 0.95, xtol=1e-4)
print(f'\nDesign requirement (inverting the model)')
print(f'  STAT6 requirement ({REQ_S6:.2f}% knockdown) is cleared above escape = {req(REQ_S6)*100:.1f}%')
print(f'  RelA  requirement ({REQ_RA:.2f}% knockdown) is cleared above escape = {req(REQ_RA)*100:.1f}%  <- binding')

## 6 · Figure 1 — compartments and RISC loading

Left: the three delivery compartments filling to steady state. Right: the gap the
document calls "the whole point" — free siRNA keeps accumulating while active RISC
saturates, because the carrier runs out before the cargo does.

In [ ]:
t = r['t'] / 24
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

a = ax[0]
a.plot(t, r['E'], color=BLUE, lw=2.4, label='extracellular pool E')
a.plot(t, r['N'], color=MID,  lw=2.0, label='endosomal N')
a.plot(t, r['B'], color=TAN,  lw=2.0, label='cytoplasmic B')
a.set_xlim(0, 14); a.set_xlabel('days'); a.set_ylabel('CO-BERA (nM)')
a.set_title('Compartments reach steady state', color=BLUE, fontsize=11.5, pad=9)
a.legend(fontsize=9, labelcolor=DARK)

b = ax[1]
b.plot(t, r['S'],     color=TAN,  lw=2.4, label='free siRNA per branch')
b.plot(t, r['A'],     color=BLUE, lw=2.6, label='active RISC per branch')
b.plot(t, r['Rfree'], color=MID,  lw=1.8, ls='--', label='free RISC capacity')
b.set_xlim(0, 14); b.set_xlabel('days'); b.set_ylabel('concentration (nM)')
b.set_title('RISC loading saturates by day 3', color=BLUE, fontsize=11.5, pad=9)
b.legend(fontsize=9, labelcolor=DARK)
b.annotate(f'{at(r,"S",336):.0f} nM', xy=(13.6, at(r,'S',336)),
           color='#a06d33', fontsize=9, ha='right', va='bottom')
b.annotate(f'{at(r,"A",336):.2f} nM', xy=(13.6, at(r,'A',336)+3),
           color=BLUE, fontsize=9, ha='right')

fig.tight_layout(); show(fig, 'QUENCH_Figure1')

## 7 · Figure 2 — transcript and protein

Both branches share one fractional knockdown, so each transcript lands at
E × (1 − f) — the same knockdown, different destinations, because they start from
different elevations.

Protein is obtained by passing the transcript through a first-order lag,
d[protein]/dt = k·(transcript − protein), with k = ln2/t½. Three half-lives are drawn
because the document is explicit that these come from broad proteome measurements,
not target-specific data for STAT6 or RELA.

In [ ]:
def protein_from(mrel, t, thalf=46.0, y0=None):
    """First-order protein lag behind a transcript trajectory."""
    kp = np.log(2) / thalf
    y0 = mrel[0] if y0 is None else y0
    s = solve_ivp(lambda tt, y: [kp * (np.interp(tt, t, mrel) - y[0])],
                  (t[0], t[-1]), [y0], method='LSODA',
                  rtol=1e-10, atol=1e-13, t_eval=t)
    return s.y[0]

t = r['t'] / 24
fig, ax = plt.subplots(1, 2, figsize=(11, 3.7))
for k, (m, lab, Efac, key, hlev) in enumerate([
        (r['m1']*E_S6, 'STAT6', E_S6, 'STAT6', S6_H),
        (r['m2']*E_RA, 'RelA',  E_RA, 'RelA',  RA_H)]):
    a = ax[k]
    a.plot(t, m*100, color=BLUE, lw=2.8, label='transcript')
    for th, st, c, lw in [(24, ':', MID, 1.5), (46, '-', TAN, 1.9), (72, '--', MID, 1.5)]:
        a.plot(t, protein_from(m, r['t'], th, y0=Efac)*100,
               color=c, ls=st, lw=lw, label=f'protein t½ {th} h')
    a.axhline(100, color=RED, ls='--', lw=1.5)
    a.text(13.7, 102, 'healthy', color=RED, fontsize=9, ha='right')
    tc = cross(r, key, hlev)
    a.axvline(tc/24, color=RED, ls=':', lw=1.4)
    a.annotate(f'crosses healthy\nat {tc:.1f} h', xy=(tc/24, 112),
               xytext=(tc/24 + 1.4, 118), color=RED, fontsize=8.6,
               arrowprops=dict(arrowstyle='-', color=RED, lw=1.0))
    a.annotate(f'settles at {m[-1]*100:.0f}%', xy=(11.5, m[-1]*100),
               xytext=(8.6, m[-1]*100 - 13), color=BLUE, fontsize=8.8,
               arrowprops=dict(arrowstyle='-', color=BLUE, lw=1.0))
    a.set_xlim(0, 14); a.set_ylim(55, 142)
    a.set_xlabel('days'); a.set_ylabel(f'{lab}, % of healthy')
    a.set_title(f'{lab} — starts {Efac*100-100:.1f}% above healthy',
                color=BLUE, fontsize=11.5, pad=9)
    a.legend(fontsize=8.2, labelcolor=DARK, loc='upper right', ncol=2)

fig.tight_layout(); show(fig, 'QUENCH_Figure2')

## 8 · Figure 3 — the secretion sweep

The saturation result. Eight times the secretion rate raises knockdown from 36.6% to
44.2% and no further, because the Argonaute pool — not the supply of siRNA — sets the
ceiling. This is why the document says coverage is capped by biology rather than dose.

In [ ]:
mult = np.logspace(np.log10(1/8), np.log10(8), 22)
pl   = np.array([plateau(alpha=ALPHA*m) for m in mult])
tR   = []
for m in mult:
    rr = run(alpha=ALPHA*m, t_end=400, n=4000)
    c  = cross(rr, 'RelA', RA_H)
    tR.append(np.nan if c is None else c)
tR = np.array(tR)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
a = ax[0]
a.semilogx(mult, pl, color=BLUE, lw=2.8, solid_capstyle='round')
a.axhline(REQ_RA, color=RED, ls='--', lw=1.6)
a.axhline(REQ_S6, color=TAN, ls='--', lw=1.6)
a.text(8, REQ_RA+1.3, f'RelA needs {REQ_RA:.1f}%',  color=RED,       fontsize=8.8, ha='right')
a.text(8, REQ_S6+1.3, f'STAT6 needs {REQ_S6:.1f}%', color='#a06d33', fontsize=8.8, ha='right')
a.plot([1], [plateau()], 'o', color=BLUE, ms=7, zorder=6)
a.annotate(f'design point\n{plateau():.1f}%', xy=(1, plateau()), xytext=(0.17, 40),
           color=BLUE, fontsize=8.8, arrowprops=dict(arrowstyle='-', color=BLUE, lw=1.0))
a.annotate(f'{pl[-1]:.1f}% at 8×', xy=(8, pl[-1]), xytext=(2.4, pl[-1]-6),
           color=BLUE, fontsize=8.8, arrowprops=dict(arrowstyle='-', color=BLUE, lw=1.0))
a.set_xlabel('secretion rate, relative to design')
a.set_ylabel('plateau knockdown (%)')
a.set_title('Eightfold more secretion buys 8 points', color=BLUE, fontsize=11.5, pad=9)

b = ax[1]
b.semilogx(mult, tR, color=BLUE, lw=2.8, solid_capstyle='round')
b.set_xlabel('secretion rate, relative to design')
b.set_ylabel('time for RelA to reach healthy (h)')
b.set_title('Time to the RelA endpoint', color=BLUE, fontsize=11.5, pad=9)

fig.tight_layout(); show(fig, 'QUENCH_Figure3')

## 9 · Severity coverage — Eq 10

With a fractional knockdown f, a transcript elevated E-fold lands at E(1−f). The
highest elevation that can be brought back to healthy is E_max = 1/(1−f).

**Read this as the document does.** E_max = 1.58 is an *analytic ceiling of the nominal
model*, not a statement that some percentage of patients is covered. The real
patient-coverage fraction needs the per-sample GSE121212 distribution, which the
document lists as an outstanding computational task.

In [ ]:
f    = plateau() / 100
Emax = 1 / (1 - f)
print(f'nominal f = {f*100:.1f}%   ->   analytic ceiling E_max = 1/(1-f) = {Emax:.3f}')
print(f'cohort-mean elevations:  STAT6 {E_S6:.4f}x   RelA {E_RA:.4f}x\n')

rows = []
for E, note in [(1.05, ''), (E_S6, 'STAT6 cohort mean'), (1.10, ''),
                (E_RA, 'RelA cohort mean'), (1.40, ''),
                (Emax, 'ceiling'), (2.00, '')]:
    after = E * (1 - f)
    rows.append(dict(E=round(E, 4), elevated=f'{(E-1)*100:.1f}%',
                     after_treatment=round(after, 3),
                     excess_corrected=f'{100*(E-1-(after-1))/(E-1):.0f}%',
                     normalised='yes' if after <= 1.0 else 'no', note=note))
print(pd.DataFrame(rows).to_string(index=False))

f8 = plateau(alpha=ALPHA*8)/100
print(f'\nAt 8x secretion: f = {f8:.3f} -> E_max = {1/(1-f8):.2f}'
      f'   (the ceiling barely moves, because Argonaute saturates)')

## 10 · Figure 4 — Scenario-A open-loop handover

The promoter is **forced** to zero at day 5. That is the Scenario-A construct, not a
behaviour the equations produce on their own, and the document keeps it as a
conditional open-loop bound.

Protein lags transcript, so the protein minimum is shallower and later than the
transcript minimum. The rise past healthy after day 10 is an artifact of the forcing:
Eq 7 returns the transcript to its lesional set point once suppression ends, because
the model contains no term for the lesion itself resolving.

In [ ]:
HANDOVER_DAY = 5.0

rh    = run(t_off=HANDOVER_DAY*24.0, t_end=336.0, n=14000)
th    = rh['t'] / 24
mrel  = rh['m2'] * E_RA
prot  = protein_from(mrel, rh['t'], 46.0, y0=E_RA)

fig, a = plt.subplots(figsize=(10, 4.2))
for x0, x1, c, lab in [(0, 1.4, '#bfe3f2', 'loading'), (1.4, 4, '#dcecf6', 'effect'),
                       (4, 6, '#fbe6d2', 'handover'), (6, 14, '#e8efe8', 'vesicle-only')]:
    a.axvspan(x0, x1, color=c, zorder=0)
    a.text((x0+x1)/2, 44, lab, ha='center', fontsize=9.5, color=DARK)
for t_half, st, c, lw in [(24, ':', MID, 1.6), (46, '-', BLUE, 2.8), (72, '--', MID, 1.6)]:
    a.plot(th, protein_from(mrel, rh['t'], t_half, y0=E_RA)*100,
           color=c, ls=st, lw=lw, label=f'protein t½ {t_half} h')
a.plot(th, mrel*100, color=GREY, ls='-.', lw=1.6, label='mRNA')
a.axhline(100, color=RED, ls='--', lw=1.6)
a.text(13.85, 101.5, 'healthy', color=RED, fontsize=9.5, ha='right')
a.axhline(50, color=GREY, ls=':', lw=1.5)
a.text(13.85, 51.5, 'RELA haploinsufficiency level', color='#8a97a1', fontsize=9, ha='right')
i = int(np.argmin(prot))
a.plot([th[i]], [prot[i]*100], 'o', color=BLUE, ms=6, zorder=6)
a.annotate(f'{prot[i]*100:.0f}% on day {th[i]:.1f}', xy=(th[i], prot[i]*100),
           xytext=(th[i]+1.5, prot[i]*100 - 9), color=BLUE, fontsize=9.5,
           arrowprops=dict(arrowstyle='-', color=BLUE, lw=1.0))
a.set_xlim(0, 14); a.set_ylim(40, 132)
a.set_xlabel('days from application'); a.set_ylabel('RelA, % of healthy baseline')
a.set_title('Scenario-A open-loop bound', color=BLUE, fontsize=12, pad=10)
a.legend(loc='upper right', ncol=2, fontsize=9, labelcolor=DARK)
fig.tight_layout(); show(fig, 'QUENCH_Figure4')

print('RelA protein minimum by assumed half-life (handover day 5):')
for t_half in (24, 46, 72):
    p = protein_from(mrel, rh['t'], t_half, y0=E_RA)
    print(f'   t½ {t_half:>2} h -> {p.min()*100:5.1f}% of healthy')

print('\nSweeping the handover day (46 h half-life):')
for d in (3, 4, 5, 6, 8, 10):
    rr = run(t_off=d*24.0, t_end=336.0, n=12000)
    mm = rr['m2']*E_RA
    pp = protein_from(mm, rr['t'], 46.0, y0=E_RA)
    j  = int(np.argmin(pp))
    print(f'   handover day {d:>2}: min {pp[j]*100:5.1f}% on day {rr["t"][j]/24:5.2f}'
          f'   day-14 {np.interp(336, rr["t"], pp)*100:6.1f}%')

## 11 · Figure 5 — the conditional closed loop

This adds **one explicit coupling assumption** that is not in the model as documented:
lesional peroxide tracks NF-κB activity between the healthy and lesional set points.
That relationship has not been measured.

The document is careful about what this does and does not show, and so should you be.
Because the coupling anchors the low-inflammatory end of the loop to the healthy set
point, convergence near 100% is partly *built into the closure*. The defensible
conclusion is narrow: the architecture **can** be self-limiting under a plausible
monotone coupling. It is not an independent prediction that the real system settles
at healthy.

An inflammatory challenge is applied around day 9 to test re-engagement.

In [ ]:
CHALLENGE_DAY, CHALLENGE_WIDTH = 9.0*24.0, 1.5*24.0

def H_closed(rrel, t, frac_heal=0.40, frac_les=0.90):
    """UNMEASURED coupling: lesional peroxide tracks NF-kB activity."""
    boost = 1.0 + 0.55*np.exp(-0.5*((t - CHALLENGE_DAY)/(CHALLENGE_WIDTH/2))**2)
    frac  = frac_heal + (frac_les - frac_heal)*np.clip((rrel - 1.0)/(E_RA - 1.0), 0, None)
    frac  = min(frac*boost, 0.97)
    return KM_NPX*(frac*VMAX_FLARE)/(VMAX_FLARE - frac*VMAX_FLARE)

def run_loop(closed, T=336.0, n=16000, t_off=5*24.0):
    qq = dict(Q)
    def f(t, y):
        rrel = max(y[14], 0)*E_RA
        if closed:
            H, gate = H_closed(rrel, t), 1.0
        else:
            H    = H_SS if t <= t_off else 0.0
            gate = 1.0  if t <= t_off else 0.0
        return rhs(t, y, H, gate, qq, ALPHA)
    y0 = [P1['PerR_tot'], 0, 0, P1['Trx_bas'], 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.0, 1.0]
    s  = solve_ivp(f, (0, T), y0, method='LSODA', rtol=1e-9, atol=1e-12,
                   t_eval=np.linspace(0, T, n))
    t, So = s.t, s.y[2]
    Pt = P1['P_basal'] + (1 - P1['P_basal'])*hill(So, P1['K_out'], P1['n_out'])
    if not closed:
        Pt = np.where(t <= t_off, Pt, 0.0)
    mrel = s.y[14]*E_RA
    return t/24.0, mrel, protein_from(mrel, t, 46.0, y0=E_RA), Pt

tc, mc, pc, Pc = run_loop(True)
to, mo, po, Po = run_loop(False)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.7))
a = ax[0]
a.axvspan(8.25, 9.75, color='#f7ece2', zorder=0)
a.text(9.0, 70, 'inflammatory\nchallenge', color='#a06d33', fontsize=8.6, ha='center')
a.plot(to, po*100, color=GREY, lw=2.0, ls='--', label='open loop (Figure 4)', zorder=3)
a.plot(tc, pc*100, color=BLUE, lw=2.8, label='closed loop + sensor', zorder=4)
a.axhline(100, color=RED, ls='--', lw=1.5)
a.text(0.3, 101.5, 'healthy', color=RED, fontsize=9)
k = int(np.argmax(np.where(tc > 6, pc, -1)))
a.annotate(f'peaks at {pc[k]*100:.0f}%\nthen returns', xy=(tc[k], pc[k]*100),
           xytext=(10.4, 113), color=BLUE, fontsize=8.8,
           arrowprops=dict(arrowstyle='-', color=BLUE, lw=1.0))
a.annotate('', xy=(13.3, 91), xytext=(13.3, 98.5),
           arrowprops=dict(arrowstyle='->', color=TAN, lw=1.6))
a.text(13.0, 86, 'EV effect\n(not modelled)', color='#a06d33', fontsize=8.4, ha='center')
a.set_xlim(0, 14); a.set_ylim(64, 130)
a.set_xlabel('days after application'); a.set_ylabel('RelA protein, % of healthy')
a.set_title('Conditional closed loop', color=BLUE, fontsize=11, pad=8)
a.legend(loc='upper left', fontsize=8.6, labelcolor=DARK)

b = ax[1]
b.axvspan(8.25, 9.75, color='#f7ece2', zorder=0)
b.fill_between(tc, 0, Pc, color=FILL, zorder=1)
b.plot(to, Po, color=GREY, lw=2.0, ls='--', zorder=3)
b.plot(tc, Pc, color=BLUE, lw=2.8, zorder=4)
b.annotate('sensor re-engages on\nthe new H$_2$O$_2$', xy=(9.4, float(np.interp(9.4, tc, Pc))),
           xytext=(3.9, 0.76), color=BLUE, fontsize=8.8,
           arrowprops=dict(arrowstyle='-', color=BLUE, lw=1.0))
b.text(6.6, 0.125, 'open loop: forced to zero at day 5', color=GREY, fontsize=8.2)
b.set_xlim(0, 14); b.set_ylim(0, 0.95)
b.set_xlabel('days after application'); b.set_ylabel('P$_{trxB}$ (promoter activity)')
b.set_title('The bacteria stay armed for 14 days', color=BLUE, fontsize=11, pad=8)

fig.tight_layout(); show(fig, 'QUENCH_Figure5')

print(f'closed loop : minimum {pc.min()*100:.1f}%   day-14 {pc[-1]*100:.1f}%'
      f'   post-challenge peak {pc[tc>6].max()*100:.1f}%')
print(f'open loop   : minimum {po.min()*100:.1f}%   day-14 {po[-1]*100:.1f}%')
print('\nThe 100.0% is conditional on the imposed coupling, not an independent result.')

## 12 · Figure 6 — local sensitivity

Each parameter halved and doubled, measured at 72 h. The baseline is 34.6%, below the
36.6% plateau, because the compartments are still filling at 72 h.

Note what this plot *cannot* show: because k₁, k₂, k₃, k_eff, a and k₄ only reach the
outcome through their three ratios, perturbing them one at a time splits their real
influence three ways. That is exactly the artifact that made an earlier version of
this analysis conclude delivery did not matter. The global analysis in the next cell
samples the efficiencies instead, and reaches the opposite conclusion.

In [ ]:
names = ['k10','k7','Rtot','kA','k6','k5','k3','k2','k_eff','k1','k4','a','k8']
base  = 100*(1 - at(run(t_end=120, n=4000), 'm1', 72.0))

rows = []
for nm in names:
    lo = 100*(1 - at(run(q={nm: Q[nm]*0.5}, t_end=120, n=4000), 'm1', 72.0)) - base
    hi = 100*(1 - at(run(q={nm: Q[nm]*2.0}, t_end=120, n=4000), 'm1', 72.0)) - base
    rows.append((nm, lo, hi))
rows.sort(key=lambda x: max(abs(x[1]), abs(x[2])))

fig, a = plt.subplots(figsize=(9, 4.6))
y = np.arange(len(rows))
a.barh(y, [x[1] for x in rows], color=TAN,  height=0.62, label='×0.5')
a.barh(y, [x[2] for x in rows], color=BLUE, height=0.62, label='×2')
a.set_yticks(y); a.set_yticklabels([x[0] for x in rows], fontsize=10)
a.axvline(0, color=DARK, lw=1.1)
a.set_xlabel(f'change in knockdown at 72 h (percentage points, base {base:.1f}%)')
a.set_title('Halving and doubling each parameter', color=BLUE, fontsize=12.5, pad=10)
a.legend(loc='lower right', fontsize=9.5, labelcolor=DARK)
fig.tight_layout(); show(fig, 'QUENCH_Figure6')

print(f'72 h baseline {base:.2f}%   (plateau is {plateau():.2f}%)')
for nm, lo, hi in rows[::-1][:5]:
    print(f'   {nm:<7} x0.5 {lo:+6.2f} pp   x2 {hi:+6.2f} pp')

## 13 · Figure 7 — global sensitivity

Ten parameters, not thirteen, because the delivery chain is exactly degenerate in three
places — so the number sampled equals the number of effective parameter combinations.
Latin hypercube, seed 20260910.

The prior ranges below are the team's stated choices, printed so the result can be
regenerated. **They are judgements, not measurements**, and the percentages are
conditional on them — the ordering (k₇ first, escape second) is the robust part.

> ⏱ This is the slow cell — about 2 minutes at `N = 1500`. Drop `N` to 300 for a quick
> look; the percentages will wobble by a few points but the ranking will not.

In [ ]:
GSA_SEED = 20260910
N_SAMPLES = 1500        # <- lower this to 300 for a fast preview

PRIORS = {
    'k7':         (0.001,  0.1,   'log'),   # calibrated, no primary source
    'k10':        (0.035,  0.17,  'log'),   # mRNA half-life 4-20 h
    'Rtot':       (5.0,    50.0,  'log'),   # available Argonaute capacity
    'k5':         (0.0001, 0.01,  'log'),   # identifiable only against Rtot
    'kA':         (0.026,  0.23,  'log'),   # active RISC turnover
    'k6':         (0.007,  0.12,  'log'),   # free siRNA degradation
    'k8':         (2.0,    30.0,  'log'),   # cleavage; verified insensitive
    'escape':     (0.01,   0.80,  'log'),   # k2/(k2+k3) -- the decisive one
    'uptake':     (0.30,   0.95,  'lin'),   # k1/(k1+k_eff), a fraction
    'processing': (0.05,   0.98,  'log'),   # a/(a+k4)
}
GNAMES = list(PRIORS)

def draw(u, pr):
    v, q = {}, {}
    for j, nm in enumerate(GNAMES):
        lo, hi, mode = pr[nm]
        v[nm] = (np.exp(np.log(lo) + u[j]*(np.log(hi) - np.log(lo)))
                 if mode == 'log' else lo + u[j]*(hi - lo))
    for k in ('k7','k10','Rtot','k5','kA','k6','k8'):
        q[k] = v[k]
    q['k3']    = Q['k2']*(1 - v['escape'])/v['escape']
    q['k_eff'] = Q['k1']*(1 - v['uptake'])/v['uptake']
    q['k4']    = Q['a'] *(1 - v['processing'])/v['processing']
    return v, q

def gsa(pr=PRIORS, seed=GSA_SEED, n=N_SAMPLES):
    U, rows = qmc.LatinHypercube(d=len(GNAMES), seed=seed).random(n), []
    for u in U:
        v, q = draw(u, pr)
        try:
            pl = 100*(1 - at(run(q=q, t_end=400.0, n=2400), 'm1', 400.0))
        except Exception:
            continue
        if not np.isfinite(pl) or pl < -1:
            continue
        v['plateau'] = pl
        rows.append(v)
    return rows

rows = gsa()
p    = np.array([x['plateau'] for x in rows])
pc   = lambda mask: 100.0*np.mean(mask)

print(f'{len(rows)} accepted of {N_SAMPLES}   median {np.median(p):.1f}%'
      f'   5-95% {np.percentile(p,5):.1f}-{np.percentile(p,95):.1f}%\n')
print(f'  STAT6 normalised   {pc(p > REQ_S6):.0f}%')
print(f'  RelA  normalised   {pc(p > REQ_RA):.0f}%')
print(f'  both  normalised   {pc(p > REQ_RA):.0f}%   (identical by construction under B5)')

keep = [x for x in rows if x['plateau'] > 0]
lp   = np.log([x['plateau'] for x in keep])
corr = sorted(((nm, spearmanr([x[nm] for x in keep], lp).statistic) for nm in GNAMES),
              key=lambda t: -abs(t[1]))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
a = ax[0]
a.hist(np.clip(p, 0, 95), bins=55, color=BLUE, alpha=.85, edgecolor='white', linewidth=.3)
a.axvline(REQ_S6, color=TAN, ls='--', lw=1.6)
a.axvline(REQ_RA, color=RED, ls='--', lw=1.6)
top = a.get_ylim()[1]
a.text(REQ_S6+2, top*.95, f'STAT6\n{REQ_S6:.1f}%', color='#a87134', fontsize=8.6, va='top')
a.text(REQ_RA+2, top*.70, f'RelA\n{REQ_RA:.1f}%',  color=RED,       fontsize=8.6, va='top')
a.set_xlabel('plateau knockdown (%)'); a.set_ylabel('accepted samples')
a.set_title(f'Plateau across the stated priors ({len(rows)} samples)',
            color=BLUE, fontsize=11, pad=9)

b  = ax[1]
cc = corr[::-1]
nm, vv = [c[0] for c in cc], [c[1] for c in cc]
b.barh(range(len(nm)), vv, color=[BLUE if v > 0 else TAN for v in vv], height=.66)
b.set_yticks(range(len(nm))); b.set_yticklabels(nm, fontsize=9)
for i, v in enumerate(vv):
    b.text(v + (.015 if v > 0 else -.015), i, f'{v:+.2f}', va='center',
           ha='left' if v > 0 else 'right', fontsize=8, color=DARK)
b.axvline(0, color=DARK, lw=1); b.set_xlim(-.72, .72)
b.set_xlabel('Spearman rank correlation with log(plateau)')
b.set_title('What sets the outcome', color=BLUE, fontsize=11, pad=9)

fig.tight_layout(); show(fig, 'QUENCH_Figure7')

print('\nRank correlation with log(plateau):')
for nm_, c in corr:
    print(f'   {nm_:<11} {c:+.2f}')

### Delivery held at its design value

The comparison that isolates the cost of the escape assumption: fix the three delivery
efficiencies at their nominal values and sample only the seven kinetic parameters.

In [ ]:
PB = dict(PRIORS)
for k, v in [('escape', 0.75), ('uptake', 0.8508), ('processing', 0.9718)]:
    PB[k] = (v*0.999, v*1.001, 'lin')

rows_k = gsa(PB, seed=99, n=max(400, N_SAMPLES//3))
pk = np.array([x['plateau'] for x in rows_k])

print(f'delivery fixed at design ({len(rows_k)} samples)')
print(f'   median plateau      {np.median(pk):.1f}%   (nominal {plateau():.1f}%)')
print(f'   RelA normalised     {100*np.mean(pk > REQ_RA):.0f}%')
print(f'\nSampling the delivery chain as well gives {100*np.mean(p > REQ_RA):.0f}%.')
print('The whole gap between those two numbers is the price of the escape assumption.')

## Notes

**Files.** Every figure is written as `QUENCH_FigureN.png` alongside the notebook —
open the folder icon in the Colab sidebar to download them. Set `SAVE_PNG = False`
in the setup cell if you only want them inline.

**Reproducibility.** The GSA uses seed 20260910 and a Latin hypercube, so re-running
gives identical numbers. Changing `N_SAMPLES` changes the sample set, so the
percentages will shift by a few points — the ranking will not.

**Two things this notebook deliberately does not do.**

It does not invent a value for the endosomal escape fraction of *L. plantarum*
extracellular vesicles. 0.75 is carried as the nominal assumption because the document
carries it, and the requirement (escape > 17%) is computed by inverting the model
rather than by asserting a baseline.

It does not add a quantitative term for the anti-inflammatory activity of the vesicles
themselves. That effect is real and documented, but no measured coupling exists, so
Figure 5 marks its direction with an arrow and leaves the magnitude out.

**If you change a parameter**, re-run the verification cell in §4 first. Thirteen
published numbers are checked there, and if a row flips to `XX` the change has moved
something the document reports.